# OpenAI APIs and Multimodal AI

A hands-on, beginner-friendly walkthrough of the OpenAI Python SDK — covering
text generation, a small personal assistant, vision (image understanding),
image generation, and image editing, all through the modern `client.responses`
and `client.images` interfaces.

```
                     ┌───────────────────────────┐
                     │        OpenAI API          │
                     │  (text · vision · images)  │
                     └─────────────┬─────────────┘
                                   │
        ┌──────────────────────────┼──────────────────────────┐
        │                          │                          │
   Text Generation            Vision Input              Image Generation
   (chat / Q&A)          (understand an image)         (create / edit images)
```

**What you'll build in this notebook:**
1. A simple text chatbot
2. A small `PersonalAssistant` class (Q&A + email summarizer)
3. An image-understanding call using the Vision input format
4. An AI-generated image, saved to disk
5. An edited version of that image (recoloring an object)

## 1. Introduction to OpenAI APIs

### What is an API?
An **API (Application Programming Interface)** is a defined way for one program to
ask another program to do something. Instead of running a huge language model on
your own laptop, you send a request over the internet to OpenAI's servers, and they
send back a response. You never touch the model weights directly — the API is the
"contract" that describes what you can ask for and what you'll get back.

### What is an API Key?
An **API key** is a secret string that identifies *you* (or your project) to OpenAI's
servers. Every request you send must include this key so OpenAI knows who to bill and
what usage limits apply. Treat it like a password:
- Never hard-code it directly in a notebook you plan to share
- Never commit it to GitHub
- Load it from an environment variable or a secrets manager instead

### How the OpenAI client works
The `openai` Python package gives you a `OpenAI` client object. Once created, that
one object exposes every capability of the API as methods — text, vision, and image
generation all go through the same client.

```
   your code            OpenAI() client            OpenAI servers
 ─────────────         ─────────────────         ─────────────────
 client.responses.create(...)  ──────────────▶   model runs your prompt
                                ◀──────────────   JSON response comes back
```

### Request and Response flow
Every call follows the same shape:

```
 REQUEST                                   RESPONSE
 ┌─────────────────────────┐               ┌─────────────────────────┐
 │ model   -> which model   │               │ output_text -> the reply│
 │ input   -> your prompt   │  ──────────▶  │ usage       -> token    │
 │ settings-> temperature,  │               │                counts   │
 │           max tokens...  │               │ id, status, etc.        │
 └─────────────────────────┘               └─────────────────────────┘
```

We'll use this exact request → response pattern for every example below, whether
we're generating text, reading an image, or creating one.

## 2. Installation

Install the official OpenAI SDK. Everything in this notebook — text, vision, and
images — is available through this single package.

In [ ]:
# Uncomment the line below the first time you run this notebook.
# The -q flag just keeps the install output quiet.
# !pip install openai -q

### Imports and client setup

We load the API key from an **environment variable** rather than typing it into the
notebook. This keeps the key out of version control and out of shared notebooks.

In [ ]:
import os
import base64
from openai import OpenAI

# The SDK will also automatically pick up an OPENAI_API_KEY environment
# variable on its own, but being explicit here makes it clear where the
# key comes from and makes it easy to swap in a different loading method
# (e.g. getpass, a secrets manager, a .env file) later.
api_key = os.environ.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("OpenAI client ready:", client is not None)

## 3. Text Generation

### Your first request
The `client.responses.create()` method is the main entry point for talking to a
model. At minimum it needs a `model` name and some `input` (your prompt).

In [ ]:
# Pick which model handles the request. Different models trade off
# speed, cost, and reasoning ability -- swap this constant to compare them.
TEXT_MODEL = "gpt-5.4"

first_response = client.responses.create(
    model=TEXT_MODEL,
    input="Write a two-line bedtime story about a curious robot.",
)

# output_text is a convenience property that already extracts and
# concatenates the plain-text portion of the response for you.
print(first_response.output_text)

### Choosing a model
Model choice is a trade-off, not a "best" answer:

| Priority | What to pick |
|---|---|
| Fastest / cheapest | a lightweight / "mini" model |
| Best reasoning on hard problems | a larger flagship model |
| Balanced everyday use | a mid-tier general-purpose model |

Start with a smaller model while you're prototyping a prompt, then upgrade only if
the answers aren't good enough — this saves both time and API cost.

### Controlling randomness with `temperature`
`temperature` controls how "creative" vs. "predictable" the output is:
- **Low temperature (0 – 0.3)** → focused, deterministic, good for facts and math
- **High temperature (0.7 – 1.0+)** → more varied, creative, good for brainstorming

### Limiting length with `max_output_tokens`
`max_output_tokens` caps how long the reply can be. A *token* is roughly ¾ of a word,
so this is a way to bound both response length and cost — without it, a model could
in theory ramble on far longer than you need.

In [ ]:
# temperature=0.2 keeps this answer focused and consistent -- good for
# a factual, single-correct-answer style question.
math_response = client.responses.create(
    model=TEXT_MODEL,
    input="Solve for x: x + 2 = 7. Show one line of working.",
    temperature=0.2,
    max_output_tokens=60,
)

print(math_response.output_text)

## 4. Personal AI Assistant

### System vs. user roles
Instead of one flat string, you can pass a **list of role-tagged messages**:

- **`system`** — instructions that shape *how* the model should behave for the whole
  conversation (its persona, tone, or constraints)
- **`user`** — the actual question or task from the person using the app

```
 [ system: "You are a concise, friendly assistant." ]
 [ user:   "What's the capital of France?"           ]
                       │
                       ▼
              model replies in that persona
```

### Prompt engineering, briefly
Prompt engineering just means *being deliberate about what you tell the model*:
- Give it a role (system message) so its tone/behavior is consistent
- Be specific about the output format you want ("in 2–3 sentences", "as a bullet list")
- Provide any context it needs (e.g. the actual email text to summarize)

Below we wrap both ideas — roles + prompt design — into a small reusable class.

In [ ]:
class PersonalAssistant:
    """A minimal AI assistant with two skills: answering questions and
    summarizing emails. Both skills reuse the same OpenAI client, but each
    sets a different system role and a different temperature, since one
    task wants precision and the other wants a natural, readable summary.
    """

    def __init__(self, client, model=TEXT_MODEL):
        self.client = client
        self.model = model
        print("Assistant online. Ask a question or hand me an email to summarize.")

    def answer_question(self, question):
        """Answer a free-form user question as a helpful general assistant."""
        response = self.client.responses.create(
            model=self.model,
            input=[
                {"role": "system", "content": "You are a helpful, concise personal assistant."},
                {"role": "user", "content": question},
            ],
            temperature=0.7,       # a little creative flexibility for open questions
            max_output_tokens=512,
        )
        return response.output_text.strip()

    def summarize_email(self, email_text):
        """Condense a longer email into a short 2-3 sentence summary."""
        instruction = f"Summarize the following email in 2-3 sentences:\n\n{email_text}"

        response = self.client.responses.create(
            model=self.model,
            input=[
                {"role": "system", "content": "You are an expert at writing crisp email summaries."},
                {"role": "user", "content": instruction},
            ],
            temperature=0.3,       # low temperature keeps summaries factual and stable
            max_output_tokens=512,
        )
        return response.output_text.strip()

In [ ]:
assistant = PersonalAssistant(client)

In [ ]:
# Try it on a direct question.
answer = assistant.answer_question("What's a good way to start learning about APIs?")
print(answer)

In [ ]:
# A sample email to summarize -- swap this out for any real email text.
sample_email = """
Hi Team,

I hope you're all doing well. I wanted to reach out to schedule a review
meeting to go over last quarter's project outcomes. We've made solid
progress on several fronts, but I'd also like to discuss a few blockers
that came up along the way and how we can address them going forward.

Could you let me know your availability sometime next week? I expect the
meeting to take about 30-45 minutes, and I'll bring a short summary of
our key metrics to guide the discussion.

Thanks in advance,
Priya
Engineering Lead
"""

summary = assistant.summarize_email(sample_email)
print("Summary:", summary)

## 5. Vision API — Understanding Images

The same `responses.create()` call can accept **images alongside text** in a single
message. Instead of a plain string, `content` becomes a list mixing two content
types:

- **`input_text`** — the written part of your prompt
- **`input_image`** — a reference to an image (here, a URL)

```
 content: [
   { type: "input_text",  text: "What is in this picture?" },
   { type: "input_image", image_url: "https://..."         }
 ]
        │
        ▼
 model reads BOTH the text and the image together
```

This lets you ask questions *about* an image, rather than just generating or editing
one.

In [ ]:
VISION_MODEL = "gpt-5.4"

# Any publicly reachable image URL works here.
image_url = "https://images.pexels.com/photos/3476860/pexels-photo-3476860.jpeg"

vision_response = client.responses.create(
    model=VISION_MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Describe what's happening in this image in one sentence.",
                },
                {
                    "type": "input_image",
                    "image_url": image_url,
                },
            ],
        }
    ],
)

print(vision_response.output_text)

## 6. Image Generation

There are two different ways to generate images with the OpenAI SDK, and this
notebook covers both. First, the **Responses API** can generate an image as a *tool*
that a model call invokes on your behalf.

### Base64 encoding and decoding, briefly
Images come back from the API as **Base64** — a way of representing binary data
(like PNG bytes) using only plain text characters. It's the same trick used to embed
images inside HTML or JSON. To turn it back into a real image file, you **decode**
it back into raw bytes and write those bytes to disk:

```
 raw PNG bytes  --encode-->  Base64 text  --(sent over the API as JSON)-->
 Base64 text    --decode-->  raw PNG bytes  --write to disk-->  image.png
```

In [ ]:
IMAGE_TOOL_MODEL = "gpt-5"

generation_response = client.responses.create(
    model=IMAGE_TOOL_MODEL,
    input="Generate a highly detailed image of a futuristic spaceship flying past a colorful nebula.",
    tools=[{"type": "image_generation"}],
)

# The response can contain multiple output items; we only want the ones
# produced by the image_generation tool call.
generated_images_b64 = [
    item.result
    for item in generation_response.output
    if item.type == "image_generation_call"
]


def save_base64_image(base64_string, filename):
    """Decode a Base64-encoded image string and write it to disk as a file."""
    image_bytes = base64.b64decode(base64_string)
    with open(filename, "wb") as image_file:
        image_file.write(image_bytes)
    print("Saved:", filename)


if generated_images_b64:
    save_base64_image(generated_images_b64[0], "spaceship_via_responses_api.png")
else:
    print("No image was returned -- check the prompt or model access.")

## 7. Dedicated Images API

Alongside the Responses API's image tool, OpenAI also offers a **standalone Images
API** (`client.images`) purpose-built for creating and editing pictures. It's a
more direct route when image generation *is* the whole task, rather than one step
inside a larger conversation.

In [ ]:
IMAGE_MODEL = "gpt-image-1.5"

image_prompt = "A highly detailed, futuristic spaceship flying near a colorful nebula, digital art."

image_result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=image_prompt,
)

# Just like before, the pixel data comes back Base64-encoded.
generated_image_b64 = image_result.data[0].b64_json

save_base64_image(generated_image_b64, "spaceship_via_images_api.png")

## 8. Image Editing

`client.images.edit()` takes an *existing* image plus an instruction, and returns a
modified version. Here we ask it to recolor the spaceship we just generated — the
original file becomes the reference image for the edit.

In [ ]:
edit_prompt = "Using the reference image, repaint the spaceship's hull in a bright red color."

with open("spaceship_via_images_api.png", "rb") as reference_image:
    edit_result = client.images.edit(
        model=IMAGE_MODEL,
        prompt=edit_prompt,
        image=[reference_image],
    )

edited_image_b64 = edit_result.data[0].b64_json

save_base64_image(edited_image_b64, "spaceship_recolored_red.png")

## 9. Conclusion

### Responses API vs. Images API — what's the difference?

| | Responses API (`client.responses`) | Images API (`client.images`) |
|---|---|---|
| Best for | Conversations that *might* include an image generation step | Tasks that are purely about creating/editing images |
| Input | Text, and optionally images (vision), plus tools | A text prompt (and, for edits, a reference image) |
| Output | Text, and optionally an image via the `image_generation` tool | Image data only (`b64_json` or a URL) |
| Mental model | "Have a conversation, maybe make an image along the way" | "I need an image, that's the whole job" |

### Real-world applications
- **Customer support bots** — text generation + a knowledge base for grounded answers
- **Accessibility tools** — Vision API to describe images for visually impaired users
- **Marketing/creative tools** — Image Generation for quick concept art or ad creatives
- **Product photo editing** — Image Editing for recoloring, background changes, variants
- **Personal productivity** — assistants like the one built above for Q&A and email triage

### Best practices
- Start with a smaller/cheaper model, upgrade only when quality demands it
- Use `system` messages to keep an assistant's behavior consistent across calls
- Set `temperature` deliberately: low for facts, higher for creative tasks
- Always cap `max_output_tokens` in production code to control cost
- Handle empty/missing results defensively (as we did with `generated_images_b64`)

### API key security
- Never hard-code a real key in a notebook, script, or commit history
- Load keys from environment variables, `.env` files (git-ignored), or a secrets manager
- Rotate a key immediately if it's ever exposed publicly
- Use separate keys per project/environment so you can revoke one without affecting others

---
*This notebook is for educational purposes as part of an AI/ML learning portfolio.*